In [ ]:
import os

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

In [ ]:
template_df = pd.read_csv("../data/mhcii_tcr_templates.csv", index_col="pdb_id")
pdb_ids = template_df.index.to_list()

In [ ]:
ensembles_dir = "../data/mhcii_tcr_ensembles/"
sampling_methods = ["static/crystal", "static/pandora2", "ensemble/annealing"]
docking_method = "haddock3_rigidbody"

In [ ]:
df = None

for pdb_id in pdb_ids:
    for sampling_method in sampling_methods:
        ensemble_dir = os.path.join(pdb_id, sampling_method, docking_method)
        csv_fp = os.path.join(ensembles_dir, ensemble_dir, "stats.csv")
        if not os.path.exists(csv_fp):
            print(csv_fp)
            continue
        ensemble_df = pd.read_csv(csv_fp, index_col="model_id")
        ensemble_df.drop("pdb_fp", axis=1, inplace=True)
        ensemble_df["pdb_id"] = pdb_id
        ensemble_df["sampling_method"] = sampling_method
        if df is None:
            df = ensemble_df
        else:
            df = pd.concat([df, ensemble_df])

df

In [ ]:
metrics = ["rmsd", "dockq", "dockq_fnat", "binding_core_sasa"]
colors = ["blue", "limegreen", "crimson", "green"]

In [ ]:
for metric in metrics:
    fig = go.Figure()
    for i, pdb_id in enumerate(pdb_ids):
        pdb_df = df[df.pdb_id == pdb_id]
        for sampling_method, color in zip(sampling_methods, colors):
            ens_df = pdb_df[pdb_df.sampling_method == sampling_method]
            fig.add_trace(
                go.Box(
                    y=ens_df[metric].values,
                    name=pdb_id,
                    offsetgroup=sampling_method,
                    legendgroup=sampling_method,
                    showlegend=(i == 0),
                    line=dict(color=color),
                    marker=dict(color=color)
                )
            )
    fig.update_xaxes(showticklabels=True)
    if metric == "dockq":
        fig.update_yaxes(range=[0, 1.1])
    elif metric == "rmsd":
        fig.update_yaxes(range=[0, 35])
    fig.update_layout(title=metric, boxmode="group")
    fig.show()

In [ ]:
for metric in metrics:
    for sampling_method in sampling_methods:
        fig = px.scatter(df[df.sampling_method == sampling_method], x="crossing_angle", y="incident_angle", color=metric, title=sampling_method)
        fig.update_xaxes(range=(0, 180))
        fig.update_yaxes(range=(0, 90))
        fig.show()